In [1]:
from torchvision.models import mobilenet_v3_large as mobilenet
from calflops import calculate_flops
import torch.nn.functional as func
import torch
import os

/home/tteuz/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONFIG = {"device": "cuda:0"}
ENSEMBLE_PATH = "../pipeline/resources/classification/CNRParkEXT/ensemble_pklot_plds_7e87d"
# ENSEMBLE_PATH = "../pipeline/resources/classification/PKLot/ensemble_cnrparkext_plds_2bff1"
# ENSEMBLE_PATH = "../pipeline/resources/classification/PLds/ensemble_pklot_cnrparkext_3f836"

SIAMESE_PATH = "../pipeline/resources/siamese/CNRParkEXT/model_pklot_plds_01e83.pt"
# SIAMESE_PATH = "../pipeline/resources/siamese/PKLot/model_cnrparkext_plds_5c4be.pt"
# SIAMESE_PATH = "../pipeline/resources/siamese/PLds/model_pklot_cnrparkext_2da05.pt"

In [3]:
NUM_CLASSES = 2

class model():
    def __init__(self, config):
        self.model = mobilenet()
        self.config = config

        self.model.classifier[-1] = torch.nn.Linear(1280, NUM_CLASSES)
        self.model = self.model.to(self.config["device"])

        for name, param in self.model.named_parameters():
            if "classifier" not in name and "features.16" not in name:
                param.requires_grad = False
        
    
    def load_state_dict(self, state_dict):
        self.model.load_state_dict(state_dict)
        self.model.to(self.config["device"])


class ensemble_model():
    def __init__(self, state_dicts, config):
        self.models = []

        for state_dict in state_dicts:
            internal_model = model(config)
            internal_model.load_state_dict(state_dict)
            self.models.append(internal_model)

In [4]:
class siamese_model(torch.nn.Module):
    def __init__(self, config=None):
        super(siamese_model, self).__init__()

        backbone = mobilenet()
        self.features = torch.nn.Sequential(*list(backbone.children())[:-1])
        self.fc = torch.nn.Linear(960, 128)
        self.l2norm = func.normalize

        self.config = config
        self.to(device=config["device"])


    # def forward(self, inputs):
    #     towers = [self.features(inp) for inp in inputs]
    #     towers = [torch.flatten(tower, 1) for tower in towers]
    #     towers = [self.fc(tower) for tower in towers]
    #     towers = [self.l2norm(tower, dim=1) for tower in towers]

    #     return towers

    def forward(self, input):
        x = self.features(input)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.l2norm(x, dim=1)

        return x

In [5]:
def format_number(num):
    suffixes = ['', 'K', 'M', 'G', 'T', 'P', 'E']
    num = float(num)
    i = 0

    while abs(num) >= 1000 and i < len(suffixes) - 1:
        num /= 1000.0
        i += 1

    return f"{num:.2f}{suffixes[i]}"

In [6]:
ENSEMBLE_INPUT_SHAPE = (1, 3, 128, 128)

classif_state_dicts = [torch.load(os.path.join(ENSEMBLE_PATH, model), map_location=CONFIG["device"]) for model in os.listdir(ENSEMBLE_PATH)]
classif_ensemble = ensemble_model(classif_state_dicts, CONFIG)

total_ensemble_flops = 0
for internal_model in classif_ensemble.models:
    internal_model.model.eval()
    
    flops, _, _ = calculate_flops(model=internal_model.model, input_shape=ENSEMBLE_INPUT_SHAPE, output_as_string=False, print_detailed=False, print_results=False)
    total_ensemble_flops += flops
    
    print(f"{format_number(flops)} FLOPs")

print(f"\nTotal Ensemble: {format_number(total_ensemble_flops)} FLOPs")

148.15M FLOPs
148.15M FLOPs
148.15M FLOPs

Total Ensemble: 444.44M FLOPs


In [7]:
SIAMESE_INPUT_SHAPE = (1, 3, 128, 128)
siamese = siamese_model(CONFIG)

siamese.eval()
total_siamese_flops = 0
for index in range(2):
    flops, _, _ = calculate_flops(siamese, SIAMESE_INPUT_SHAPE, output_as_string=False, print_detailed=False, print_results=False)

    total_siamese_flops += flops
    print(f"{format_number(flops)} FLOPs")

print(f"\nTotal Siamese: {format_number(total_siamese_flops)} FLOPs")

145.93M FLOPs
145.93M FLOPs

Total Siamese: 291.86M FLOPs


In [8]:
total_flops = total_ensemble_flops + total_siamese_flops

print(f"Total necessary: {format_number(total_flops)} FLOPs")

Total necessary: 736.30M FLOPs
